<a href="https://colab.research.google.com/github/jeffRnR/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeffRnR/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why
Lane 2: Refresh / Content Opportunity Scoring.

With 30,000 pages across multiple clients, no content team can review everything, they need a ranked list that tells them where to look first. This lane builds exaclty thatL a scoring system that combines observable search signals to surface pages most worth a human reviewer's time. The outpyt is a prioritized queue with reason codes, not a black-box score. I chose this lane because the core problem, turning noizy signals into a ranked action list, is the same pattern I am building in a couple of other personal projects.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Dataset: {df.shape[0]:,} pages, {df.shape[1]} columns")
print(f"Clients represented: {df['client_id'].nunique()}")
print(f"Declining pages: {(df['trend_direction'] == 'down').sum():,} ({(df['trend_direction'] == 'down').mean():.1%})")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 2. The question: decision, action, cost of a wrong call

**Question:** Which pages should a content reviewer look at first for refresh,
protection, or pruning — given limited review capacity?

**Unit of analysis:** One page (one content item) per scoring run.

**Decision it improves:** A content team decides which pages to spend time on this
week. Without a ranked list, they guess or use recency alone.

**Action someone takes:** Open the flagged page, read the reason code
(e.g. "stale visible page" or "declining with demand"), and decide whether to
rewrite, expand, protect, or prune it.

**Cost of a wrong recommendation:**
- False positive (flagging a healthy page): wasted reviewer time on a page that
  did not need work.
- False negative (missing a page that needed work): a page continues losing
  visibility and traffic with no intervention.

For a small content team, false negatives are more expensive — a missed page keeps
declining silently. That is why Precision@50 is the right metric: it measures
whether the pages the system surfaces most urgently are actually the ones worth fixing.

In [ ]:
declining = df[df["trend_direction"] == "down"]
stable    = df[df["trend_direction"] == "stable"]

print(f"Declining pages — median impressions_90d: {declining['impressions_90d'].median():,.0f}")
print(f"Stable pages   — median impressions_90d: {stable['impressions_90d'].median():,.0f}")
print(f"\nDeclining pages with impressions >= 500 (high-stakes): {(declining['impressions_90d'] >= 500).sum():,}")
print("These are pages with real audience exposure that are actively losing ground.")

Declining pages — median impressions_90d: 961
Stable pages   — median impressions_90d: 1,944

Declining pages with impressions >= 500 (high-stakes): 9,961
These are pages with real audience exposure that are actively losing ground.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset that make this lane worth 7 weeks:

1. The random forest in Notebook 01 got 37 of its top 50 recommendations right
   (Precision@50: 0.740) versus 12 for the hand-written rule (0.240) — a 3x improvement.
   That gap is the opportunity this lane closes.

2. Correlation between search_volume and impressions_90d is 0.001 — near zero.
   Simple rules built on keyword volume alone will mis-rank most pages. A learned
   model that combines multiple signals does better.

3. The numbers below show how many high-visibility pages are actively declining —
   these are the pages where a wrong prioritization decision has the highest cost.

In [ ]:
print("=== Signal 1: Learned model vs hand-written rule ===")
print("Baseline rule  Precision@50: 0.240  (~12 of top 50 correct)")
print("Random forest  Precision@50: 0.740  (~37 of top 50 correct)")
print("Improvement: 3.1x")
print("Source: outputs/model_results.json from starter pipeline (Notebook 01)\n")

print("=== Signal 2: Search volume barely predicts actual traffic ===")
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation — search_volume vs impressions_90d: {corr:.3f}")
print("Near zero — keyword volume alone cannot reliably rank pages for review.\n")

print("=== Signal 3: Scale of high-stakes declining pages ===")
high_stakes = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 500)]
print(f"Total pages: {len(df):,}")
print(f"Declining pages: {(df['trend_direction'] == 'down').sum():,} ({(df['trend_direction'] == 'down').mean():.1%})")
print(f"High-visibility declining pages (impressions >= 500): {len(high_stakes):,}")
print(f"Median traffic drop on those pages: {high_stakes['trend_pct'].median():.1f}%")
print("A content team cannot manually triage all of these — a ranked list is essential.")

=== Signal 1: Learned model vs hand-written rule ===
Baseline rule  Precision@50: 0.240  (~12 of top 50 correct)
Random forest  Precision@50: 0.740  (~37 of top 50 correct)
Improvement: 3.1x
Source: outputs/model_results.json from starter pipeline (Notebook 01)

=== Signal 2: Search volume barely predicts actual traffic ===
Correlation — search_volume vs impressions_90d: 0.001
Near zero — keyword volume alone cannot reliably rank pages for review.

=== Signal 3: Scale of high-stakes declining pages ===
Total pages: 30,000
Declining pages: 16,262 (54.2%)
High-visibility declining pages (impressions >= 500): 9,961
Median traffic drop on those pages: -49.7%
A content team cannot manually triage all of these — a ranked list is essential.


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- "Pages with these observable signals were associated with declining traffic
  in this dataset." (observed, directional)
- "This ranked list surfaces declining pages more reliably than a hand-written
  rule, measured by Precision@50 on a client-holdout validation." (decision-support)
- "A reviewer checking the top 50 pages from this system will find more
  genuine decline candidates than by using volume or staleness alone." (measured)

**What this work will never claim:**
- That refreshing a flagged page will cause traffic to recover — this data cannot
  prove causation, only association.
- That this predicts Google's algorithm or ranking factors.
- That a high score guarantees a page needs work — reason codes exist so a human
  reviewer can apply judgment.
- That results on the 30,000-row starter slice will hold exactly on the full
  79M-row warehouse — that has to be earned again with proper validation.## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
label_dist = df["trend_direction"].value_counts(normalize=True)
print("Trend direction distribution (this is our proxy label, not a guaranteed outcome):")
print(label_dist.round(3).to_string())
print("\nNote: 'down' means recent trend is declining — not that the page is permanently broken.")
print("A reviewer must still apply judgment before acting on any recommendation.")

Trend direction distribution (this is our proxy label, not a guaranteed outcome):
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038

Note: 'down' means recent trend is declining — not that the page is permanently broken.
A reviewer must still apply judgment before acting on any recommendation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.